# 02 — Data Quality

Audit only: keep every annotation. No cleaning, feature engineering, POS or dataset split. Review candidates do not imply wrong labels.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src/data/data_quality.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook with the kernel working directory inside the project.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import json
from src.data.constants import SGML_FILE, ANNOTATIONS_FILE
from src.data.nucle_sgml import load_nucle
from src.data.parse_nucle import documents_to_dataframe
from src.data.data_quality import audit_quality, build_report

documents = load_nucle(SGML_FILE)
annotations = documents_to_dataframe(documents)
summary, audit, duplicates = audit_quality(documents, annotations, ANNOTATIONS_FILE)
print(build_report(summary))


## Source/CSV consistency and quality flags

Missing correction tags differ from empty corrections. Flags preserve original offsets and labels.


In [ ]:
print(json.dumps(summary, indent=2))
print(audit.loc[audit.review_status == "pending"].head(20).to_string(index=False))


## Review annotation context

Each annotation has its own ID and label. Check whether overlapping spans represent distinct errors, duplicate annotations or different labels on the same region. Do not automatically merge or remove them.


In [ ]:
review = annotations.merge(
    audit[["annotation_id", "review_reasons", "overlap_annotation_ids"]],
    on="annotation_id", validate="one_to_one",
)
for label, group in review[review.is_target_label].groupby("error_type"):
    print("\n" + label)
    print(group.sample(min(4, len(group)), random_state=42)[
        ["annotation_id", "essay_id", "paragraph", "error_span", "correction",
         "review_reasons", "overlap_annotation_ids"]
    ].to_string(index=False))


## Duplicate text

Report exact/normalized essay and paragraph duplicates; do not create groups or folds. Short boilerplate and paraphrases require manual review.


In [ ]:
print(json.dumps(duplicates[:10], indent=2))
assert len(audit) == len(annotations)
assert audit.annotation_id.tolist() == annotations.annotation_id.tolist()
print("All annotation rows and IDs preserved.")


## Save the reports

Run `python -m src.data.data_quality` from the repository. Notebook cells only read and display; future data preparation is outside the current scope.
